In [1]:
import pandas as pd
from symspellpy import SymSpell, Verbosity
from itertools import islice
import pkg_resources
import re

In [2]:
gender = pd.read_csv('lai-data/gender.csv')

frequency_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(frequency_path, term_index=0, count_index=1)

bigram_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_bigramdictionary_en_243_342.txt")
sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

In [24]:
def correct_freq_dict(text):
    
    # get all words in english
    words = re.findall(r'\b[a-zA-Z]+\b', text)
    corrected_words = []
    
    for word in words:
        # find best correction candidate
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        corrected_word = suggestions[0].term if suggestions else word
        corrected_words.append(corrected_word)
    
    # replace misspells 
    corrected_text = text
    for original, corrected in zip(words, corrected_words):
        corrected_text = re.sub(r'\b{}\b'.format(re.escape(original)), corrected, corrected_text, count=1)
    
    return corrected_text


def correct_bigram_dict(text):

    words = re.findall(r'\b[a-zA-Z]+\b', text)
    
    # join words into a sentence to get context
    english_part = " ".join(words)
    
    # find best correction candidate 
    suggestions = sym_spell.lookup_compound(english_part, max_edit_distance=2)
    corrected_text = suggestions[0].term if suggestions else english_part
    
    # replace misspells 
    corrected_words = corrected_text.split()
    for original, corrected in zip(words, corrected_words):
        text = re.sub(r'\b{}\b'.format(re.escape(original)), corrected, text, count=1)
    
    return text

In [25]:
gender_small = gender.loc[:5]
gender_small['corrected_freq_dict'] = gender_small['post'].apply(correct_freq_dict)
gender_small['corrected_bigram_dict'] = gender_small['post'].apply(correct_bigram_dict)

C:\Users\Goshko\anaconda3\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
C:\Users\Goshko\anaconda3\lib\site-packages\ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


In [29]:
# gender_small.loc[5].corrected_freq_dict

In [30]:
# gender_small.loc[5].corrected_bigram_dict

In [31]:
# gender_small.loc[5].post

In [ ]:
# gender['corrected_freq_dict'] = gender['post'].apply(correct_freq_dict)
# gender['corrected_bigram_dict'] = gender['post'].apply(correct_bigram_dict)

# gender.to_csv("spelling_corrected_gender.csv", index=False)